[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/01_build_your_agent.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a>

# Build a Financial Agent

Build a tool-calling agent over a synthetic bank ledger, then watch it fail silently.
Run cells from top to bottom. Enter your workshop key when prompted.


---
## Connect

Install, then point a client at the workshop gateway with your seat key.


In [ ]:
%pip install -q "openai>=1.40" "pandas>=2.0" "matplotlib>=3.8"


In [ ]:
from getpass import getpass
from openai import OpenAI

GATEWAY_URL = "https://fmlds-eval-workshop-gateway.fly.dev/v1"
AGENT_MODEL = "gpt-4o-mini"

client = OpenAI(api_key=getpass("Workshop API key: ").strip(), base_url=GATEWAY_URL)
client.chat.completions.create(model=AGENT_MODEL, messages=[{"role": "user", "content": "ping"}], max_tokens=2)
print("Gateway connected")


---
## The data: a synthetic bank ledger

2,500 synthetic transactions from May–October 2026. Amounts are signed: income and refunds
are positive; expenses are negative. There is no location column.

| column | meaning | trap |
|---|---|---|
| `date` | transaction date | |
| `raw_description` | raw bank string: `STARBUCKS #1042`, `AMZN Mktp US*1A2B3C` | string fragmentation |
| `category` | 10 broad categories | |
| `amount` | **signed**: positive = income/refund, negative = expense | polarity inversion |
| `account_type` | Checking / Savings / Credit Card | |


In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/data/transactions.csv"
df = pd.read_csv(DATA_URL)
df["date"] = pd.to_datetime(df["date"])

print(f"loaded {len(df)} transactions")
df.head()


---
## Tool 1 - Inspect the ledger

This function reports the available columns and shows sample rows. It does not assume
that a particular column exists.


In [ ]:
def fetch_data_schema():
    """Return available columns, row count, and sample rows."""
    return {
        "columns": {column: str(dtype) for column, dtype in df.dtypes.items()},
        "rows": len(df),
        "sample_rows": df.head(5).astype(str).to_dict(orient="records"),
    }


In [ ]:
fetch_data_schema()

---
## Tool 2 - Analyze the transactions

This tool has two steps: generate pandas code, then run it. The router calls one
high-level tool; these cells show what happens inside it.


First, ask the model to write pandas code using the ledger schema.


In [ ]:
ANALYSIS_CODE_PROMPT = """Write concise pandas code to answer the question.
Use the existing DataFrame df. Print the final result.
Columns: {columns}
Sample rows: {sample}
Question: {question}
Return Python code only. Do not import libraries."""


def generate_analysis_code(question):
    """Ask the model to write pandas code for a ledger question."""
    schema = fetch_data_schema()
    prompt = ANALYSIS_CODE_PROMPT.format(
        columns=schema["columns"], sample=schema["sample_rows"], question=question,
    )
    response = client.chat.completions.create(
        model=AGENT_MODEL, messages=[{"role": "user", "content": prompt}],
    )
    code = response.choices[0].message.content.replace("```python", "").replace("```", "").strip()
    print("Generated pandas code:")
    print(code)
    return code


Next, run the code. The ledger is available as `df`; printed output and errors go back to the model.


In [ ]:
import io
from contextlib import redirect_stdout

import matplotlib.pyplot as plt

_AGENT_NS = {}


def execute_python_code(code_str):
    """Run code on df and return printed output or an error."""
    _AGENT_NS.update({"df": df, "pd": pd, "plt": plt})
    output = io.StringIO()
    try:
        with redirect_stdout(output):
            exec(code_str, _AGENT_NS)
        ok = True
        error = ""
    except Exception as e:
        ok = False
        error = f"{type(e).__name__}: {e}"
    if ok and plt.get_fignums():
        plt.show()
    plt.close("all")
    return {"ok": ok, "stdout": output.getvalue()[-2000:], "stderr": error[-800:]}


In [ ]:
execute_python_code("print(df['amount'].sum())")

Finally, this function connects code generation and execution as one tool call.


In [ ]:
def analyze_transactions(question):
    """Generate and run pandas code to answer a ledger question."""
    code = generate_analysis_code(question)
    result = execute_python_code(code)
    result["code"] = code
    return result


In [ ]:
analyze_transactions("What were my expenses in October?")

---
## Tool 3 - Look up a financial rule

This function returns a stored rule when its keywords match the question.


In [ ]:
FINANCIAL_RULES = [
    {"keywords": ["50/30/20", "budget"], "rule": "50/30/20: spend up to 50% on needs, 30% on wants, and save at least 20% of after-tax income."},
    {"keywords": ["emergency", "fund", "buffer"], "rule": "Keep 3-6 months of essential expenses in liquid savings."},
    {"keywords": ["debt", "loan"], "rule": "Keep monthly debt payments below 36% of gross monthly income."},
    {"keywords": ["housing", "rent", "mortgage"], "rule": "Keep housing costs below 30% of gross monthly income."},
    {"keywords": ["saving", "savings"], "rule": "Aim to save at least 20% of income."},
    {"keywords": ["subscription", "recurring"], "rule": "Review recurring subscriptions regularly and cancel unused ones."},
]


def financial_rules_lookup(query):
    """Return finance rules whose keywords appear in the question."""
    q = query.lower()
    return {"matches": [r["rule"] for r in FINANCIAL_RULES if any(word in q for word in r["keywords"])]}


In [ ]:
financial_rules_lookup("How does the 50/30/20 budget work?")

---
## Tool 4 - Visualize the transactions

First ask the model for chart code using the ledger columns and sample rows. Then run
that code against `df` and display the chart.


In [ ]:
CHART_PROMPT = """Write short Python code to make the requested chart.
Use the existing pandas DataFrame df and matplotlib.pyplot as plt.
Columns: {columns}
Sample rows: {sample}
Chart goal: {goal}
Return Python code only. Do not import libraries."""


def generate_chart_code(visualization_goal):
    """Ask the model for Python code that makes the requested chart."""
    schema = fetch_data_schema()
    prompt = CHART_PROMPT.format(
        columns=list(schema["columns"]), sample=schema["sample_rows"], goal=visualization_goal,
    )
    response = client.chat.completions.create(
        model=AGENT_MODEL, messages=[{"role": "user", "content": prompt}],
    )
    code = response.choices[0].message.content.replace("```python", "").replace("```", "").strip()
    print("Generated chart code:")
    print(code)
    return code


The next function connects chart generation to the Python executor, so the router can
call visualization as one tool. The test below makes one live model call and displays
the chart.


In [ ]:
def visualize_transactions(visualization_goal):
    """Generate and display a chart of the transaction ledger."""
    code = generate_chart_code(visualization_goal)
    return execute_python_code(code)


In [ ]:
visualize_transactions("Bar chart of spending by category")

---
## Describe the tools to the model

The schema tells the model each tool's name, purpose, and arguments.


In [ ]:
TOOL_SPECS = [
    {"type": "function", "function": {
        "name": "fetch_data_schema",
        "description": "Show ledger columns and sample rows.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }},
    {"type": "function", "function": {
        "name": "analyze_transactions",
        "description": "Answer a transaction question with pandas code.",
        "parameters": {"type": "object", "properties": {"question": {"type": "string"}}, "required": ["question"]},
    }},
    {"type": "function", "function": {
        "name": "financial_rules_lookup",
        "description": "Look up a personal-finance rule.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "visualize_transactions",
        "description": "Create a chart from the transaction ledger.",
        "parameters": {"type": "object", "properties": {"visualization_goal": {"type": "string"}}, "required": ["visualization_goal"]},
    }},
]


The implementation map connects each schema name to its Python function.


In [ ]:
TOOLS = {
    "fetch_data_schema": fetch_data_schema,
    "analyze_transactions": analyze_transactions,
    "financial_rules_lookup": financial_rules_lookup,
    "visualize_transactions": visualize_transactions,
}


---
## The system prompt

We start with the kind most people actually ship - one warm, vague sentence. That is our
baseline "vibe-coded" agent. (The hardened and creative prompts arrive in notebook 3.)


In [ ]:
NAIVE_SYSTEM_PROMPT = """You are a helpful data analysis assistant. Answer the user's questions about their personal finances using the provided tools."""

print("baseline prompt: one warm, vague sentence - the kind most people actually ship")


---
## The agent loop - hand-rolled, no framework

The model chooses tools from their schemas. We run each call, return its result, and
repeat until the model answers or reaches the six-call limit.


Set a response limit before the agent makes its model call.


In [ ]:
import time  # trace-hook
MAX_TOKENS = 1500


`dispatch_tool` runs the Python function selected by the model and returns its result.


In [ ]:
import json

MAX_TOOL_CALLS = 6


def dispatch_tool(tc):
    """Run the selected tool and return its result or error."""
    fn = TOOLS.get(tc.function.name)
    if fn is None:
        return {"error": f"unknown tool: {tc.function.name}"}
    try:
        arguments = json.loads(tc.function.arguments or "{}")
        return fn(**arguments)
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


`run_agent` asks the model what to do, runs requested tools, and returns the final answer.


In [ ]:
def run_agent(query, system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0, model=None):
    """Run the tool-calling loop until the model returns a final answer."""
    model = model or AGENT_MODEL
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": query}]
    tool_calls_used = 0
    answer = ""
    while True:
        print("Router call")
        resp = client.chat.completions.create(
            model=model, messages=messages, temperature=temperature,
            max_tokens=MAX_TOKENS, tools=TOOL_SPECS,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls or tool_calls_used >= MAX_TOOL_CALLS:
            answer = msg.content or ""
            print("Router result:", answer)
            break
        for tc in msg.tool_calls:
            tool_calls_used += 1
            print(f"Calling {tc.function.name} tool")
            out = dispatch_tool(tc)
            print("Tool result:", out)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(out)[:4000]})
        if tool_calls_used >= MAX_TOOL_CALLS:
            messages.append({"role": "user", "content": "Tool budget exhausted - give your best final answer now."})
    result = {"answer": answer}
    return result

---
## Failure #1 - Polarity inversion (sign handling)

Ask the naive agent to use its Python tool, but do not tell it how signed amounts work.
Income and refunds are **positive**; expenses are **negative**.

Before running: what do *you* expect it to say?


In [ ]:
demo1 = run_agent("Use analyze_transactions to calculate my total expenses in October and report the dollar amount.",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0)
print(demo1["answer"])


In [ ]:
truth1 = -df.loc[(df["date"].dt.month == 10) & (df["amount"] < 0), "amount"].sum()
classic_mistake = df.loc[df["date"].dt.month == 10, "amount"].sum()
print(f"\ntruth (sum of negative October amounts):   {truth1:,.2f}")
print(f"the signed total of all October amounts:    {classic_mistake:,.2f}")
print(f"income and refunds included in that total:  {df.loc[(df['date'].dt.month == 10) & (df['amount'] > 0), 'amount'].sum():,.2f}")
print("\nThe signed total is net cash flow, not expense magnitude.")


### Harder sign case: net expenses after refunds

Gross expenses count negative transactions. Net expenses also subtract refunds, but must
not subtract income. The ledger contains an October refund for this case.


In [ ]:
demo1b = run_agent("Use analyze_transactions to calculate net expenses in October after transaction refunds, excluding income.",
                               system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0)
print(demo1b["answer"])


In [ ]:
october = df[df["date"].dt.month == 10]
gross_expenses = -october.loc[october["amount"] < 0, "amount"].sum()
refunds = october.loc[
    october["raw_description"].str.contains("REFUND", case=False) & (october["amount"] > 0), "amount"
].sum()
print(f"gross expenses: {gross_expenses:,.2f}")
print(f"refunds:        {refunds:,.2f}")
print(f"net expenses:   {gross_expenses - refunds:,.2f}")


---
## Failure #2 - Merchant aliases

"How much did I spend at Starbucks?" Some descriptions contain `STARBUCKS`; others use
the alias `SBUX`. `TULLY'S COFFEE` is a decoy: coffee does not necessarily mean Starbucks.


In [ ]:
demo2 = run_agent("Use analyze_transactions to find my total Starbucks spending and report the dollar amount, even if it is zero.",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0)
print(demo2["answer"])


In [ ]:
is_starbucks = df["raw_description"].str.contains(r"STARBUCKS|SBUX", case=False, regex=True)
truth2 = -df.loc[is_starbucks & (df["amount"] < 0), "amount"].sum()
print(f"\ntruth (STARBUCKS and SBUX descriptions): {truth2:,.2f}")
print("\nthe variants hiding in raw_description:")
print(df.loc[is_starbucks, "raw_description"].value_counts().to_string())
print("\nSimilar coffee-shop spend, not Starbucks:")
print(df.loc[df["raw_description"].str.contains("TULLY'S", case=False), "raw_description"].value_counts().to_string())


---
## Failure #3 - Schema hallucination

"Show me spending by merchant location." There is **no location column**. A confident
model may invent `merchant_city` and fabricate a breakdown. Compare its answer with the
columns that are actually available.


In [ ]:
demo3 = run_agent("Show me my spending broken down by merchant location.",
                              system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0)
print(demo3["answer"])


In [ ]:

print("\nactual columns in the schema:")
print(sorted(df.columns.tolist()))
print("\nschema the agent should have consulted (fetch_data_schema):")
import json as _j
print(_j.dumps({k: v for k, v in fetch_data_schema().items() if k != "sample_rows"}, indent=2))


---
## Why "vibe-checking" fails

All three failures share one property: **the agent never crashed where the user could
see it.** Fluent, confident prose - with the bug living in the *semantics*: a swallowed
sign, an over-strict string match, an invented column. Semantics are exactly what human
eyeballs are bad at checking.

The rest of the workshop: **observe** what happened (notebook 2), **assert** what must
be true (notebook 3), **gate** on it in CI (notebook 3).

*(The code you just built also lives in the repo's canonical `utils.py`, generated from
this notebook's source. Notebook 2 imports it so you don't re-run this notebook.)*
